In [1]:
import random
import string
import numpy as np
from milvus import default_server

from pymilvus import (
    utility,
    FieldSchema, CollectionSchema, DataType,
    Collection, AnnSearchRequest, RRFRanker, connections,
)
from pymilvus.model.hybrid import BGEM3EmbeddingFunction
from FlagEmbedding import BGEM3FlagModel

In [3]:
from pymilvus import connections

connections.connect("default", host="localhost", port="19530")

if connections.has_connection("default"):
    print("Successfully connected to Milvus")
else:
    print("Failed to connect to Milvus")


Successfully connected to Milvus


In [8]:
from pymilvus import connections, list_collections, drop_collection

# Connect to Milvus server
connections.connect(alias="default", host="localhost", port="19530")

# List all collections in Milvus
collections = list_collections()
print(f"Collections in Milvus: {collections}")

# # Iterate over each collection and delete it
# for collection in collections:
#     print(f"Dropping collection: {collection}")
#     drop_collection(collection)

# print("All collections deleted.")


Collections in Milvus: ['hybrid_experiment', 'sbert_experiment_test', 'sbert_experiment']


In [9]:
# Define the data schema for the new Collection
fields = [
    # Use provided id as primary key
    FieldSchema(name="pk", dtype=DataType.VARCHAR, is_primary=True, max_length=1000),
    # Store the original text
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=65535),
    # Store dense vectors
    FieldSchema(name="dense_vector", dtype=DataType.FLOAT_VECTOR, dim=1024),  # Ensure the dimension matches your embeddings
    # Store sparse vectors
    FieldSchema(name="sparse_vector", dtype=DataType.SPARSE_FLOAT_VECTOR),  
]

schema = CollectionSchema(fields,  enable_dynamic_field=False)
col_name = 'hybrid_experiment2'
col = Collection(col_name, schema, consistency_level="Strong")

In [10]:

sparse_index = {"index_type": "SPARSE_INVERTED_INDEX", "metric_type": "IP"}
col.create_index("sparse_vector", sparse_index)
dense_index = {"index_type": "FLAT", "metric_type": "IP"}
col.create_index("dense_vector", dense_index)
col.load()

KeyboardInterrupt: 

In [ ]:
ef = BGEM3EmbeddingFunction(use_fp16=True, device='cuda')
# ef2 = BGEM3EmbeddingFunction(use_fp16=False, device='cpu', return_sparse=True, return_dense=True, return_colbert_vecs=False)
ef3 = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)

In [7]:
def merge_text_fields(data):
    for item in data:
        # Merge the fields into 'text', separating them by " | "
        merged_text = " | ".join(filter(None, [item.get('text', ''), 
                                            item.get('provided_data', ''), 
                                            item.get('enriched_data', ''), 
                                            item.get('translated_data', '')]))
        
        # Assign the merged text back to the 'text' field
        item['text'] = merged_text
        
        # Remove the individual fields as they're now part of 'text'
        item.pop('provided_data', None)
        item.pop('enriched_data', None)
        item.pop('translated_data', None)
    
    return data

def hybrid_embeddings(batch_data):
    data = merge_text_fields(batch_data)
    only_text = [x['text'] for x in data]

    # Generate embeddings using BGEM3 model
    embeddings = ef3.encode(only_text, return_sparse=True, return_dense=False, return_colbert_vecs=False)
    # embeddings = ef(only_text)
    # Debug prints to verify the dimensions
    print(f"Number of texts: {len(only_text)}")
    print(f"Dense embeddings shape: {len(embeddings['dense_vecs'])}")
    print(f"Sparse embeddings shape: {len(embeddings['lexical_weights'])}")

    # Prepare data for insertion
    entities = [
        [item['id'] for item in data],  # IDs
        only_text,  # Texts
        embeddings["dense_vecs"],  # Dense vectors
        embeddings["lexical_weights"]  # Sparse vectors
    ]

    del embeddings

    return entities

In [3]:
import os
import gzip
import json
import time
import tqdm
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed

# Function to load data from a compressed JSON file
def load_compressed_json(file_path):
    """Function to load data from a compressed JSON file."""
    with gzip.open(file_path, 'rt', encoding='utf-8') as f:
        return json.load(f)

# Function to load data in batches of 1000 documents for Milvus indexing
def load_data_in_batches_for_indexing(parsed_directory, batch_size=100, max_workers=4):
    """Load data from compressed JSON files in batches for indexing into Milvus."""
    batch_data = []  # To store the current batch
    start_time = time.time()  # Record the start time

    # Collect all the .json.gz file paths
    file_paths = []
    for root, dirs, files in os.walk(parsed_directory):
        for file in files:
            if file.endswith('.json.gz'):
                file_path = os.path.join(root, file)
                file_paths.append(file_path)

    total_files = len(file_paths)
    checkpoint_times = {}  # Dictionary to save checkpoint times

    # Use ThreadPoolExecutor to load files in parallel
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_file = {executor.submit(load_compressed_json, file_path): file_path for file_path in file_paths}

        for idx, future in enumerate(tqdm.tqdm(as_completed(future_to_file), total=total_files, desc="Loading and batching files")):
            file_path = future_to_file[future]
            try:
                data = future.result()  # Load the file's data
                
                # Add the data to the current batch
                batch_data.extend(data)  # Assuming data is a list of documents

                # If the current batch exceeds the batch size, process it
                if len(batch_data) >= batch_size:
                    # Call your Milvus insertion function here
                    index_batch_into_milvus(batch_data[:batch_size])  # Process 1000 documents at a time

                    # Remove the processed documents from the batch
                    batch_data = batch_data[batch_size:]

                # Record the time at each 10% completion
                percentage_complete = ((idx + 1) / total_files) * 100
                if percentage_complete >= 10 and (int(percentage_complete) % 10 == 0) and (int(percentage_complete) not in checkpoint_times):
                    elapsed_time = time.time() - start_time
                    checkpoint_times[int(percentage_complete)] = elapsed_time
                    print(f"Indexed {int(percentage_complete)}% of documents in {elapsed_time:.2f} seconds")

            except Exception as e:
                print(f"Error loading file {file_path}: {e}")

    # Process any remaining data in the last batch
    if batch_data:
        print(batch_data)
        index_batch_into_milvus(batch_data)  # Replace this with your Milvus indexing function

    # Save checkpoint times to a file for future use
    with open('checkpoint_times.json', 'w') as f:
        json.dump(checkpoint_times, f)
    print("Checkpoint times saved to 'checkpoint_times.json'.")

def index_batch_into_milvus(batch_data):
    print(f"Indexing batch with {len(batch_data)} documents into Milvus...")

    entities = hybrid_embeddings(batch_data)

    # Verify the lengths of each component to ensure they match
    print(f"Length of IDs: {len(entities[0])}")
    # print(f"Length of texts: {len(entities[1])}")
    # print(f"Shape of dense vectors: {len(entities[2])}")

    col.insert(entities)
    col.flush()

    del entities

    print("Batch indexed successfully.")

In [8]:
# run the function to load data in batches
load_data_in_batches_for_indexing('/home/sbasir/Thesis/Thesis/cp', max_batch_size_mb=64, max_workers=4)

Loading and batching files: 100%|██████████| 1/1 [00:00<00:00, 103.77it/s]

Indexing batch with 705 documents into Milvus...


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Inference Embeddings: 100%|██████████| 45/45 [00:10<00:00,  4.13it/s]


ValueError: Output dtype not compatible with inputs.